In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_INPUTS = True
REUSE_DENSE_SCAN = False
REUSE_REACQUISITION = False
REUSE_BRIDGE = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Dense Source-CCTA Reacquisition v1.1

Memory-safe rerun of the source-only dense distal reacquisition experiment. Scientific thresholds are unchanged. Orthogonal planes are sampled directly from the int16 source CT memmap into float32 planes, avoiding whole-volume conversion on each plane.


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -q --depth 1 --branch secondary-dense-source-reacquisition-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.secondary_dense_source_reacquisition import synthetic_dense_source_self_test
from openplaque.secondary_dense_source_reacquisition_v2 import (
    SecondaryDenseSourceReacquisitionWorkflow,
    synthetic_memmap_plane_self_test,
)

t1 = synthetic_dense_source_self_test()
t2 = synthetic_memmap_plane_self_test()
display(t1); display(t2)
assert t1['passed'], t1
assert t2['passed'], t2

wf = SecondaryDenseSourceReacquisitionWorkflow(
    root='/content/drive/MyDrive/OpenPlaque',
    reuse={
        'inputs': REUSE_INPUTS,
        'dense_scan': REUSE_DENSE_SCAN,
        'reacquisition': REUSE_REACQUISITION,
        'bridge': REUSE_BRIDGE,
        'figures': REUSE_FIGURES,
        'report': REUSE_REPORT,
    },
)
display(wf.cache_status())


In [ ]:
snapshot = wf.load_inputs(reacquisition_origin_arc_mm=12.20)
display(snapshot)
display(wf.calibration)
print('Reacquisition origin z,y,x:', wf.reacquisition_origin)
print('Origin arc:', wf.reacquisition_origin_arc, 'mm')


In [ ]:
hits = wf.dense_scan(
    longitudinal_step_mm=0.45,
    lateral_step_mm=0.55,
    max_center_hits=180,
    nms_mm=0.55,
)
display(wf.scan_summary)
print('Compact center-plane hits after NMS:', len(hits))
display(hits.head(30))


In [ ]:
tracks = wf.search_reacquisition(top_center_hits=120)
print('Tracklet candidates tested:', len(tracks))
print('Supported compact-lumen tracklets:', int(tracks.tracklet_gate.sum()) if len(tracks) else 0)
display(tracks.head(30))
if wf.best_tracklet_qc is not None and len(wf.best_tracklet_qc):
    display(wf.best_tracklet_qc)


In [ ]:
bridge = wf.assess_bridge()
print('Bridge candidates:', len(bridge))
display(bridge)


In [ ]:
summary = wf.run()
display(summary)


In [ ]:
names = wf.make_figures()
for name in names:
    path = str(wf.out / name)
    print(name)
    display(Image(filename=path))


In [ ]:
zip_path = wf.package()
print('Final ZIP:', zip_path)
print('Report:', wf.out / 'OPENPLAQUE_SECONDARY_DENSE_SOURCE_REACQUISITION_REPORT.html')
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_SECONDARY_DENSE_SOURCE_REACQUISITION_REPORT_BACK.zip')
